# NES via Bayesian sampling

Build a pool of independently trained classifiers, derive a validation-loss
posterior, and select members with MC sampling or SVGD-inspired refinement.
This is the library's **discrete pool-based** implementation, not the original
paper's weight-sharing supernet. See [setup instructions](../README.md).
All data are generated locally, and computation uses CPU.

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from bensemble.metrics import negative_log_likelihood
from bensemble.search import NESBayesianSampler, SearchSpace

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)  # Small CPU workloads avoid thread-pool overhead.
DEVICE = torch.device("cpu")
EPOCHS = 180
POOL_SIZE = 8
ENSEMBLE_SIZE = 3

## 1. Separate candidate training, selection, and evaluation

Standardization uses training statistics only. Validation labels define the
candidate posterior and selection; test labels are used only for final metrics.

In [ ]:
# Generate two noisy crescents using only PyTorch.
generator = torch.Generator().manual_seed(SEED)
n = 900
labels = torch.arange(n) % 2
angle = torch.rand(n, generator=generator) * torch.pi
features = torch.stack((angle.cos(), angle.sin()), dim=1)
features[labels == 1, 0] = 1 - features[labels == 1, 0]
features[labels == 1, 1] = 0.5 - features[labels == 1, 1]
features += 0.25 * torch.randn(n, 2, generator=generator)

indices = torch.randperm(n, generator=generator)
split_indices = {
    "train": indices[:540],
    "validation": indices[540:720],
    "test": indices[720:],
}
x_train, y_train = features[split_indices["train"]], labels[split_indices["train"]]
x_val, y_val = (
    features[split_indices["validation"]],
    labels[split_indices["validation"]],
)
x_test, y_test = features[split_indices["test"]], labels[split_indices["test"]]

# Fit preprocessing on training data only, then reuse it on the held-out sets.
train_mean = x_train.mean(dim=0)
train_std = x_train.std(dim=0).clamp_min(1e-6)
x_train = (x_train - train_mean) / train_std
x_val = (x_val - train_mean) / train_std
x_test = (x_test - train_mean) / train_std

plt.figure(figsize=(6, 4))
plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, cmap="coolwarm", s=12)
plt.xlabel("Standardized x1")
plt.ylabel("Standardized x2")
plt.title("Training data")
plt.show()

## 2. Define the space and candidate training

The space varies hidden width and depth. All models are trained independently
using the same number of steps and the same training split.

In [ ]:
class ToySearchSpace(SearchSpace):
    WIDTHS = (16, 32, 64)
    DEPTHS = (1, 2)

    def sample(self):
        return {
            "width": random.choice(self.WIDTHS),
            "depth": random.choice(self.DEPTHS),
        }

    def mutate(self, config):
        child = dict(config)
        key = random.choice(["width", "depth"])
        choices = self.WIDTHS if key == "width" else self.DEPTHS
        child[key] = random.choice([value for value in choices if value != config[key]])
        return child

    def build(self, config):
        layers = []
        in_features = 2
        for _ in range(config["depth"]):
            layers.extend([nn.Linear(in_features, config["width"]), nn.ReLU()])
            in_features = config["width"]
        layers.append(nn.Linear(in_features, 2))
        return nn.Sequential(*layers)


def train_fn(model):
    model.to(DEVICE).train()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(EPOCHS):
        optimizer.zero_grad()
        loss = nn.functional.cross_entropy(model(x_train), y_train)
        loss.backward()
        optimizer.step()
    model.eval()


val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=64, shuffle=False)

## 3. Select with MC and SVGD-inspired refinement

Each public sampling call trains a **new pool**. Resetting both random seeds before
each call recreates the same candidate architectures and training randomness in
this CPU example, making the selection comparison easier to interpret. It still
trains two pools, and one seed is not enough to rank the methods reliably.

In [ ]:
sampler = NESBayesianSampler(
    space=ToySearchSpace(),
    train_fn=train_fn,
    pool_size=POOL_SIZE,
    ensemble_size=ENSEMBLE_SIZE,
    temperature=0.8,
    diversity_weight=0.8,
    svgd_steps=6,
    svgd_lr=0.25,
    device=DEVICE,
)

random.seed(SEED)
torch.manual_seed(SEED)
ensemble_mc = sampler.sample_mc(val_loader).eval()
random.seed(SEED)
torch.manual_seed(SEED)
ensemble_svgd = sampler.sample_svgd(val_loader).eval()

## 4. Evaluate with the same aggregation used for selection

The default classification criterion averages per-member probabilities. Use that
same predictive mixture for test metrics and decision boundaries. Applying softmax
after averaging logits would define a different predictor.

In [ ]:
def predict_probs(ensemble, x):
    with torch.no_grad():
        return ensemble.predict_members(x).softmax(dim=-1).mean(dim=0)


test_probabilities = {}
for name, ensemble in [("MC", ensemble_mc), ("SVGD-inspired", ensemble_svgd)]:
    probs = predict_probs(ensemble, x_test)
    test_probabilities[name] = probs
    assert ensemble.num_members == ENSEMBLE_SIZE
    assert probs.shape == (len(x_test), 2)
    assert torch.isfinite(probs).all()
    accuracy = (probs.argmax(dim=-1) == y_test).float().mean().item()
    print(
        f"{name}: accuracy={accuracy:.3f}, NLL={negative_log_likelihood(probs, y_test):.3f}"
    )

xx, yy = torch.meshgrid(
    torch.linspace(-3, 3, 100), torch.linspace(-3, 3, 100), indexing="xy"
)
grid = torch.stack([xx.flatten(), yy.flatten()], dim=-1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for ax, ensemble, title in zip(
    axes, [ensemble_mc, ensemble_svgd], ["MC", "SVGD-inspired"]
):
    probs = predict_probs(ensemble, grid)[:, 1].reshape(xx.shape)
    contour = ax.contourf(
        xx, yy, probs, levels=torch.linspace(0, 1, 11), cmap="coolwarm", alpha=0.7
    )
    ax.scatter(
        x_test[:, 0], x_test[:, 1], c=y_test, cmap="coolwarm", edgecolor="k", s=12
    )
    ax.set(xlabel="Standardized x1", ylabel="Standardized x2", title=title)
fig.colorbar(contour, ax=axes, label="P(class = 1)")
plt.show()

Try a larger pool or different temperatures using validation data, then evaluate once on untouched test data. For method comparisons across seeds, use the [benchmarks](../../benchmarks/README.md).